# Test 2 - Wine Quality

In [ ]:
from sklearn.datasets import load_wine

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
from models import KNNClassifier
from core import GridSearchCV
from preprocessing import train_test_split, normalize
import visuals as vis

In [ ]:
wine = load_wine()
X, Y = wine.data, wine.target

In [ ]:
vis.style(style='darkgrid')
vis.plot_class_distribution(Y, chart='bars', inverse_transform=lambda i: wine.target_names[i], figsize=(10, 6))

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y)

In [ ]:
z_score = normalize.z_score()

X_train = z_score.fit_transform(X_train)
X_test  = z_score.transform(X_test)

In [ ]:


# its 2 choose 13 = 55 plots, might take a while to render
vis.plot_feature_relationships(
    X_train, Y_train, 
    n_columns=5,
    feature_labels=wine.feature_names
)

In [ ]:
voting_weight = lambda d: 1 / (d + 1e-8) ** 2

grid = GridSearchCV(
    estimator=KNNClassifier(voting_weight=voting_weight),
    param_grid={"k": [1, 5, 30]},
    calculate_metrics=["confusion_matrix"]
)

grid.fit(X_train, Y_train)

for k, acc in zip(grid.cv_results_["param_k"], grid.cv_results_["mean_test_score"]):
    print(f"k = {k}: {round(acc, 4)}")

In [ ]:
vis.plot_confusion_matrices(
    grid.cv_results_["metric_confusion_matrix"],
    wine.target_names,
    titling=lambda i: f"Confusion Matrix for K = {grid.cv_results_["param_k"][i]}"
)

In [ ]:
probs = grid.predict_proba(X_test)
vis.plot_confidence_distribution(Y_test, grid.predict(X_test), probs)